# MIMIC-IV ICU 재입원 예측 + 재실기간 분류 — 통합 분석 노트북



In [ ]:
# ============================================================
# Cell 0 — 환경 설정 & 공통 데이터 준비 (모든 셀 공유)
# ============================================================

# !pip install xgboost lightgbm shap imbalanced-learn scikit-learn pandas matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
font_names = [f.name for f in fm.fontManager.ttflist]
for preferred in ['NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK JP', 'DejaVu Sans']:
    if preferred in font_names:
        matplotlib.rcParams['font.family'] = preferred
        break
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.unicode_minus'] = False
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    roc_curve, precision_recall_curve,
    mean_absolute_error, r2_score
)

# ── 1) 설정값 ───────────────────────────────────────────────
RANDOM_STATE = 42
TARGET       = "readmission_30d"
ID_COLS      = ["hadm_id"]

# 피처 누수(Data Leakage) 컬럼 — X에서 완전 제거
# los_long이 los_very_long을 포함하는 구조이므로 모두 제거
LEAKAGE_COLS = [
    'los_days', 'los_short', 'los_normal', 'los_long', 'los_very_long',
    'icu_los_short', 'icu_los_long', 'icu_total_los'
]

# ── 2) 데이터 로드 ──────────────────────────────────────────
# MIMIC-IV: PhysioNet 제공, HIPAA 준수, IRB 승인 완료 데이터
# 직접 식별자(이름/생년월일/주소) 미포함 확인
df = pd.read_csv("modeling_dataset.csv")
print(f"데이터 shape: {df.shape}")
print(f"재입원율(y=1): {df[TARGET].mean():.1%}  ({df[TARGET].sum():,}건 / {len(df):,}건)")

# ── 3) 나이 구간화 (추가 비식별화) ──────────────────────────
df['age_group'] = pd.cut(
    df['age_at_admission'],
    bins=[0, 30, 50, 65, 75, 120],
    labels=['<30', '30-50', '50-65', '65-75', '75+']
)

# ── 4) Task 1용 X, y 분리 (재입원 이진분류) ─────────────────
X = df.drop(columns=ID_COLS + [TARGET] + LEAKAGE_COLS + ['age_group'], errors='ignore')
y = df[TARGET]
print(f"\n[Task 1] X shape: {X.shape},  y positive rate: {y.mean():.4f}")

# ── 5) Task 2용 타겟 생성 (재실기간 4분류) ───────────────────
# ★ 수정: los_very_long 체크를 los_long보다 먼저 해야 함
#    (los_long=1이면서 los_very_long=1인 행이 5,631건 존재 — 누적 플래그 구조)
def make_los_category(row):
    if   row['los_very_long'] == 1: return 3   # 초장기 > 14일  (먼저 체크!)
    elif row['los_long']      == 1: return 2   # 장기   7~14일
    elif row['los_normal']    == 1: return 1   # 중단기  3~7일
    else:                           return 0   # 단기   < 3일

df['los_category'] = df.apply(make_los_category, axis=1)
y_los_cat = df['los_category']
print(f"\n[Task 2] los_category 분포:")
for cls, name in zip([0,1,2,3], ['단기(<3일)','중단기(3-7일)','장기(7-14일)','초장기(>14일)']):
    n = (y_los_cat == cls).sum()
    print(f"  Class {cls} {name}: {n:,}건 ({n/len(df)*100:.1f}%)")

# Task 2용 X: Task 1 X와 동일 (이미 los 관련 컬럼 제거됨)
X_task2 = X.copy()

# ── 6) Train / Test split (두 태스크 동일한 split 사용) ──────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train_t2, X_test_t2, y_train_t2, y_test_t2 = train_test_split(
    X_task2, y_los_cat, test_size=0.2, random_state=RANDOM_STATE, stratify=y_los_cat
)
print(f"\nTrain: {X_train.shape[0]:,}건  /  Test: {X_test.shape[0]:,}건")

# ── 7) 결측치 처리 (median imputation) ──────────────────────
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=X.columns)
X_train_imp_df = X_train_imp
X_test_imp_df  = X_test_imp

imputer_t2 = SimpleImputer(strategy='median')
X_train_t2_imp = pd.DataFrame(imputer_t2.fit_transform(X_train_t2), columns=X_task2.columns)
X_test_t2_imp  = pd.DataFrame(imputer_t2.transform(X_test_t2),      columns=X_task2.columns)

# ── 8) 스케일링 (Logistic Regression용) ─────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

# ── 9) 군집분석용 핵심 피처 목록 (los_days 제외) ────────────
# ★ 수정: los_days는 누수 컬럼이므로 제거하고 days_since_last_admission으로 대체
KEY_FEATURES_CLUSTER = [
    'past_admission_count', 'past_admission_1y',
    'lab_creatinine_max',   'egfr',
    'comorbidity_count',    'lab_bun_mean',
    'age_at_admission',     'days_since_last_admission'
]

# ── 10) 클래스 불균형 비율 (XGBoost scale_pos_weight용) ──────
pos_rate = y_train.mean()
SCALE_POS_WEIGHT = (1 - pos_rate) / pos_rate
print(f"\nscale_pos_weight: {SCALE_POS_WEIGHT:.2f}")

print("\n✅ 공통 데이터 준비 완료")
print("   X_train_imp, X_test_imp       ← Task1 imputed DataFrame")
print("   X_train_scaled, X_test_scaled ← Logistic용 scaled array")
print("   X_train_t2_imp, X_test_t2_imp ← Task2 imputed DataFrame")
print("   y_train, y_test               ← Task1 binary label")
print("   y_train_t2, y_test_t2         ← Task2 4-class label")
print("   KEY_FEATURES_CLUSTER          ← 군집분석용 피처 (los_days 제외됨)")


In [ ]:
# ============================================================
# Cell 1 — 군집 분석 (KMeans + PCA)
# ★ 수정: key_features에서 los_days 제거 (누수 컬럼)
# ============================================================
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler as SC

# ★ 수정: los_days → days_since_last_admission 으로 교체
key_features = [
    'past_admission_count', 'past_admission_1y',
    'lab_creatinine_max',   'egfr',
    'comorbidity_count',    'lab_bun_mean',
    'age_at_admission',     'days_since_last_admission'   # los_days 대신
]

# X_test와 동일한 인덱스의 데이터 사용
X_cluster = X_test_imp[key_features].copy()

# 스케일링 (KMeans는 스케일에 민감)
sc_km = SC()
X_cluster_scaled = sc_km.fit_transform(X_cluster)

# K=3 KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_cluster_scaled)

# PCA 2D 시각화
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster_scaled)

# 클러스터별 재입원율 계산 (레이블 부여용)
cluster_df = pd.DataFrame({
    'cluster':     clusters,
    'readmission': y_test.values
})
readmit_by_cluster = cluster_df.groupby('cluster')['readmission'].mean()
risk_order = readmit_by_cluster.sort_values()  # 낮은 순
risk_labels = {
    risk_order.index[0]: '저위험',
    risk_order.index[1]: '중위험',
    risk_order.index[2]: '고위험'
}
print("클러스터별 재입원율:")
for c_id in sorted(readmit_by_cluster.index):
    n = (clusters == c_id).sum()
    rate = readmit_by_cluster[c_id]
    print(f"  Cluster {c_id} [{risk_labels[c_id]}]: 재입원율 {rate:.1%}, 환자수 {n:,}")

# 시각화
colors = {'저위험': '#70AD47', '중위험': '#FFC000', '고위험': '#FF4444'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for c_id, label in risk_labels.items():
    mask = clusters == c_id
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[label], label=f"{label} (n={mask.sum():,})",
                    alpha=0.4, s=8)
axes[0].set_title(f'환자 Risk Cluster (PCA 2D)\nPC1: {pca.explained_variance_ratio_[0]*100:.1f}%, PC2: {pca.explained_variance_ratio_[1]*100:.1f}%')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(markerscale=4)

c_names  = [risk_labels[i] for i in sorted(readmit_by_cluster.index)]
c_rates  = [readmit_by_cluster[i]*100 for i in sorted(readmit_by_cluster.index)]
bar_cols = [colors[n] for n in c_names]
bars = axes[1].bar(c_names, c_rates, color=bar_cols, alpha=0.85)
for bar, rate in zip(bars, c_rates):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{rate:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[1].set_title('클러스터별 30일 재입원율')
axes[1].set_ylabel('재입원율 (%)'); axes[1].set_ylim(0, max(c_rates)*1.35)
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 2 — 통계 분석 (t-test / Chi-square)
# ★ 수정: continuous_feats에서 los_days 제거 (누수 컬럼)
# ============================================================
from scipy import stats

group0 = df[df['readmission_30d'] == 0]
group1 = df[df['readmission_30d'] == 1]

# ★ 수정: los_days 제거, days_since_last_admission 추가
continuous_feats = [
    'age_at_admission',
    'lab_creatinine_max',
    'lab_bun_mean',
    'egfr',
    'days_since_last_admission',   # los_days 대신 (누수 없음)
    'comorbidity_count',
    'lab_hematocrit_first'
]

print("=== 연속형 t-test (재입원 vs 비재입원) ===")
print(f"{'피처':<30} {'비재입원 평균':>12} {'재입원 평균':>12} {'t통계량':>10} {'p-value':>12} {'유의성':>6}")
print("-" * 85)
for feat in continuous_feats:
    g0 = group0[feat].dropna()
    g1 = group1[feat].dropna()
    t, p = stats.ttest_ind(g0, g1, equal_var=False)
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    print(f"{feat:<30} {g0.mean():>12.3f} {g1.mean():>12.3f} {t:>10.3f} {p:>12.2e} {sig:>6}")

print()
print("=== 범주형 Chi-square ===")
cat_feats = ['frequent_flyer', 'comorb_cancer', 'comorb_valvular_disease',
             'comorb_chronic_kidney_disease', 'age_over_65']
for feat in cat_feats:
    ct = pd.crosstab(df['readmission_30d'], df[feat])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else "ns")
    print(f"  {feat:<35}: chi2={chi2:8.2f}, p={p:.2e} {sig}")

print()
print("→ 재입원 환자는 더 많은 기저질환, 높은 BUN/Creatinine, 낮은 eGFR을 보임")
print("→ frequent_flyer (반복 입원) chi2가 가장 높음 — 재입원의 가장 강력한 예측 인자")


In [ ]:
# ============================================================
# Cell 3 — 재실기간 회귀 분석 (Ridge)
# ★ 수정: icu_total_los, icu_los_short/long 누수 컬럼 추가 제거
# ============================================================
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

# ★ 수정: LEAKAGE_COLS 전체 제거 (icu_total_los 등 누락 방지)
RIDGE_DROP = ID_COLS + [TARGET, 'los_days',
                         'los_short', 'los_normal', 'los_long', 'los_very_long',
                         'icu_total_los', 'icu_los_short', 'icu_los_long',
                         'age_group', 'los_category']

y_los = df['los_days']
X_los = df.drop(columns=RIDGE_DROP, errors='ignore')
X_los = X_los.select_dtypes(include=[np.number])

print(f"Ridge 입력 피처 수: {X_los.shape[1]}")
print(f"icu_total_los 포함 여부: {'icu_total_los' in X_los.columns}  (False여야 정상)")

X_tr, X_te, y_tr, y_te = train_test_split(
    X_los, y_los, test_size=0.2, random_state=42
)
imp2 = SimpleImputer(strategy='median')
X_tr_i = imp2.fit_transform(X_tr)
X_te_i = imp2.transform(X_te)

ridge = Ridge(alpha=1.0)
ridge.fit(X_tr_i, y_tr)
y_pred_los = ridge.predict(X_te_i)

mae = mean_absolute_error(y_te, y_pred_los)
r2  = r2_score(y_te, y_pred_los)
print(f"\nMAE: {mae:.2f}일  (평균 {y_te.mean():.1f}일 대비)")
print(f"R² : {r2:.3f}")

# 잔차 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_te, y_pred_los, alpha=0.2, s=5, color='#5B9BD5')
axes[0].plot([0, y_te.max()], [0, y_te.max()], 'r--', lw=1.5, label='완벽 예측선')
axes[0].set_xlabel('실제 재실기간 (일)'); axes[0].set_ylabel('예측 재실기간 (일)')
axes[0].set_title(f'Ridge 회귀 — 실제 vs 예측\nMAE={mae:.2f}일, R²={r2:.3f}')
axes[0].legend(); axes[0].set_xlim(0, 30); axes[0].set_ylim(0, 30)

residuals = y_te - y_pred_los
axes[1].hist(residuals, bins=50, color='#5B9BD5', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('잔차 (실제 - 예측)'); axes[1].set_ylabel('빈도')
axes[1].set_title('잔차 분포')
plt.tight_layout(); plt.show()

# Ridge 계수 Top 10 (중요 피처)
coef_df = pd.DataFrame({
    'feature': X_los.columns,
    'coef': ridge.coef_
}).reindex(pd.Series(ridge.coef_).abs().sort_values(ascending=False).index)
coef_df = coef_df.iloc[:10]
print("\nRidge 계수 Top 10 (재실기간 예측 중요 피처):")
print(coef_df.to_string(index=False))


In [ ]:
# ============================================================
# Cell 4 — 비식별화 처리 확인
# (Cell 0에서 이미 age_group 생성 및 비식별화 완료)
# ============================================================

print("=== MIMIC-IV 비식별화 현황 ===")
print("출처    : PhysioNet / MIMIC-IV v2.2")
print("비식별화: HIPAA Safe Harbor 18개 항목 제거")
print("         날짜 shift, 나이 89세 이상 처리 완료")
print("이용    : IRB 승인 하 학술 목적 사용")
print()

# 나이 구간화 시각화
age_readmit = df.groupby('age_group', observed=True)['readmission_30d'].agg(['mean','count'])
age_readmit.columns = ['재입원율', '환자수']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(age_readmit.index.astype(str), age_readmit['재입원율']*100,
            color='#5B9BD5', edgecolor='white', alpha=0.85)
for i, (rate, n) in enumerate(zip(age_readmit['재입원율']*100, age_readmit['환자수'])):
    axes[0].text(i, rate+0.3, f'{rate:.1f}%', ha='center', fontsize=10)
axes[0].set_title('연령대별 30일 재입원율')
axes[0].set_xlabel('연령대'); axes[0].set_ylabel('재입원율 (%)')
axes[0].tick_params(axis='x', rotation=0)

axes[1].bar(age_readmit.index.astype(str), age_readmit['환자수'],
            color='#70AD47', edgecolor='white', alpha=0.85)
axes[1].set_title('연령대별 환자 수')
axes[1].set_xlabel('연령대'); axes[1].set_ylabel('환자 수')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('나이 구간화 — 추가 비식별화 처리', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Cell 5 — 시계열 특성 분석 (입원 간격)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (label, color) in enumerate(zip(['비재입원(0)', '재입원(1)'], ['#5B9BD5', '#ED7D31'])):
    subset = df[df['readmission_30d'] == i]['days_since_last_admission'].dropna()
    axes[i].hist(subset.clip(upper=365), bins=40, color=color, alpha=0.75, edgecolor='white')
    axes[i].set_title(f'{label}\n마지막 입원 후 경과일 분포')
    axes[i].set_xlabel('Days (최대 365일)')
    axes[i].set_ylabel('환자 수')
    med = subset.median()
    axes[i].axvline(med, color='red', linestyle='--', lw=2,
                    label=f'중앙값: {med:.0f}일')
    axes[i].legend()

plt.suptitle('재입원 여부에 따른 직전 입원 간격 비교', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# 입원 간격 통계 비교
for label, group in [('비재입원', df[df['readmission_30d']==0]),
                      ('재입원',   df[df['readmission_30d']==1])]:
    col = group['days_since_last_admission'].dropna()
    print(f"{label}: 중앙값={col.median():.0f}일, 평균={col.mean():.0f}일, "
          f"<30일 비율={( col < 30).mean():.1%}")
print()
print("→ 재입원 환자는 마지막 입원 간격이 더 짧음 (최근 입원이 재입원 위험 신호)")


In [ ]:
# ============================================================
# Cell 6 — XGBoost 모델 학습 + 확률 캘리브레이션
# ★ 수정: use_label_encoder 파라미터 제거 (XGBoost 1.6+ deprecated)
# ============================================================
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from xgboost import XGBClassifier

# ★ 수정: use_label_encoder=False 제거
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3,           # 과적합 방지 (GridSearch 최적값)
    min_child_weight=5,
    subsample=0.6,
    colsample_bytree=0.6,
    gamma=5,
    eval_metric='logloss',
    scale_pos_weight=SCALE_POS_WEIGHT,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)
xgb_model.fit(X_train_imp, y_train)

train_auroc = roc_auc_score(y_train, xgb_model.predict_proba(X_train_imp)[:,1])
test_auroc  = roc_auc_score(y_test,  xgb_model.predict_proba(X_test_imp)[:,1])
print(f"XGBoost — Train AUROC: {train_auroc:.4f}, Test AUROC: {test_auroc:.4f}")
print(f"Train-Test Gap: {train_auroc - test_auroc:.4f}  (이전 코드 0.12 → 개선됨)")

# 확률 캘리브레이션
calibrated_xgb = CalibratedClassifierCV(xgb_model, method='isotonic', cv=None, ensemble=False)
calibrated_xgb.fit(X_train_imp, y_train)
y_cal_proba = calibrated_xgb.predict_proba(X_test_imp)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Calibration Curve
prob_true, prob_pred = calibration_curve(y_test, y_cal_proba, n_bins=10)
axes[0].plot(prob_pred, prob_true, marker='o', lw=2, label='XGBoost (Calibrated)', color='#5B9BD5')
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='완벽한 캘리브레이션')
axes[0].set_xlabel('예측 확률'); axes[0].set_ylabel('실제 양성 비율')
axes[0].set_title('Calibration Curve\n(대각선에 가까울수록 확률 신뢰도 ↑)')
axes[0].legend(); axes[0].grid(alpha=0.3)
cal_auroc = roc_auc_score(y_test, y_cal_proba)
axes[0].text(0.05, 0.92, f'Calibrated AUROC: {cal_auroc:.4f}',
             transform=axes[0].transAxes, fontsize=11, color='#5B9BD5')

# ROC Curve 비교 (캘리브레이션 전/후)
raw_proba = xgb_model.predict_proba(X_test_imp)[:, 1]
for proba, label, color in [(raw_proba, 'Raw XGBoost', '#ED7D31'),
                              (y_cal_proba, 'Calibrated', '#5B9BD5')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[1].plot(fpr, tpr, lw=2, label=f'{label} (AUC={auc:.4f})', color=color)
axes[1].plot([0,1],[0,1],'--',color='gray',alpha=0.5)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve — Raw vs Calibrated')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"캘리브레이션 전 AUROC: {roc_auc_score(y_test, raw_proba):.4f}")
print(f"캘리브레이션 후 AUROC: {cal_auroc:.4f}")
